# 05 Forecast backtest: three models, three test years

Can rain help forecast next weeks' national leptospirosis cases? We compare three models, each trained only on
years before the test year and tested on 2021, 2022 and 2023 in turn (rolling origin):

1. **Baseline: seasonal pattern.** Negative binomial with two seasonal harmonics and a year effect.
2. **Seasonal + rain.** The baseline plus the rain signal 2 to 6 weeks earlier (the model used in the app).
3. **Gradient boosting.** A different approach: boosted trees (Poisson loss) on week of year and the same rain lags.

**No peeking at the test year.** The year's overall case level is not known in advance, so every forecast is
scaled to recent cases: the ratio of actual to predicted cases over the 8 weeks that end 2 weeks before the
target week (DOH reports arrive about 2 weeks late). An earlier version scaled to the test year's own average,
which used test data; this notebook replaces it.

Needs only `data/processed/weekly_national.csv` (from notebook 01), so it runs without the raw files.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingRegressor
warnings.filterwarnings('ignore')

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
OUT = ROOT / 'data' / 'processed'
FC_LAGS = range(2, 7)
DELAY, WINDOW = 2, 8          # cases known up to 2 weeks before the target; 8-week scaling window
TEST_YEARS = [2021, 2022, 2023]

w = pd.read_csv(OUT / 'weekly_national.csv').sort_values(['iso_year', 'iso_week']).reset_index(drop=True)
for L in FC_LAGS:
    w[f'share_l{L}'] = w['share_prov_above_p90'].shift(L) * 10   # per 10 percentage points
for k in (1, 2):
    w[f's{k}'] = np.sin(2 * np.pi * k * w['iso_week'] / 52)
    w[f'c{k}'] = np.cos(2 * np.pi * k * w['iso_week'] / 52)
d = w.dropna(subset=['cases'])
RAIN = ' + '.join(f'share_l{L}' for L in FC_LAGS)
BASE = 'cases ~ s1 + c1 + s2 + c2 + C(iso_year)'
GBM_X = ['iso_week'] + [f'share_l{L}' for L in FC_LAGS]
MODELS = ['Baseline: seasonal', 'Seasonal + rain', 'Gradient boosting']
d.groupby('iso_year')['cases'].agg(weeks='count', mean_per_week='mean').round(1)

,weeks,mean_per_week
iso_year,,
2019,52,63.0
2020,28,51.0
2021,52,31.1
2022,52,91.8
2023,43,128.2


In [2]:
def raw_predictions(train, frame):
    """Unscaled predictions of each model for every week in `frame`, using the last training year's level."""
    last = train['iso_year'].max()
    out = pd.DataFrame(index=frame.index)
    for name, f in [(MODELS[0], BASE), (MODELS[1], f'{BASE} + {RAIN}')]:
        m = smf.negativebinomial(f, train).fit(disp=0, maxiter=500)
        out[name] = m.predict(frame.assign(iso_year=last))
    rel = train['cases'] / train.groupby('iso_year')['cases'].transform('mean')
    gbm = HistGradientBoostingRegressor(loss='poisson', max_iter=200, learning_rate=0.05, max_depth=3,
                                        random_state=0).fit(train[GBM_X], rel)
    out[MODELS[2]] = gbm.predict(frame[GBM_X].fillna(0)) * train.loc[train['iso_year'] == last, 'cases'].mean()
    return out


def scale_to_recent(frame, raw, idx):
    """Scale each target week's prediction by actual / predicted cases in the 8 weeks ending 2 weeks before it."""
    rows = []
    for i in idx:
        win = frame.loc[i - DELAY - WINDOW + 1:i - DELAY]
        win = win[win['cases'].notna()]
        row = {}
        for name in MODELS:
            r = win['cases'].sum() / raw.loc[win.index, name].sum() if len(win) >= 4 else 1.0
            row[name] = raw.loc[i, name] * r
        rows.append(row)
    return pd.DataFrame(rows, index=idx)


series, scores = [], []
for year in TEST_YEARS:
    train = d[d['iso_year'] < year]
    frame = w[w['iso_year'] <= year]
    raw = raw_predictions(train, frame)
    idx = frame.index[(frame['iso_year'] == year) & frame['cases'].notna()]
    pred = scale_to_recent(frame, raw, idx)
    y = frame.loc[idx, 'cases']
    after_rain = frame.loc[idx, [f'share_l{L}' for L in FC_LAGS]].max(axis=1) >= 2.5
    for name in MODELS:
        err = (pred[name] - y).abs()
        scores.append({'test_year': year, 'model': name, 'weeks': len(y), 'corr': np.corrcoef(pred[name], y)[0, 1],
                       'mae': err.mean(), 'mae_after_heavy_rain': err[after_rain].mean(),
                       'weeks_after_heavy_rain': int(after_rain.sum())})
    series.append(pd.concat([frame.loc[idx, ['week_start', 'cases']], pred.round(1)], axis=1).assign(test_year=year))

scores = pd.DataFrame(scores)
pooled = (scores.assign(abs_err=scores['mae'] * scores['weeks'])
          .groupby('model', sort=False).agg(weeks=('weeks', 'sum'), abs_err=('abs_err', 'sum')))
pooled['mae'] = pooled['abs_err'] / pooled['weeks']
scores.round(3).to_csv(OUT / 'forecast_scores.csv', index=False)
pd.concat(series).to_csv(OUT / 'forecast_backtest.csv', index=False)
print(pooled[['weeks', 'mae']].round(1))
scores.pivot(index='model', columns='test_year', values=['corr', 'mae']).round(2)

                    weeks   mae
model                          
Baseline: seasonal    147  46.0
Seasonal + rain       147  44.9
Gradient boosting     147  40.8


corr                mae              
test_year           2021  2022  2023   2021   2022   2023
model                                                    
Baseline: seasonal  0.11  0.57  0.68  31.73  57.30  49.59
Gradient boosting   0.05  0.68  0.75  30.88  50.73  40.74
Seasonal + rain     0.17  0.38  0.83  30.00  64.59  39.20

**Reading the result.** Both models that use rain had a lower pooled error than the seasonal baseline (gradient boosting about 41 cases a week, seasonal + rain about 45, baseline about 46), but the gain is small and not steady: seasonal + rain won in 2021 and 2023 and lost in 2022, while gradient boosting won in 2022. With about 230 weeks of national data, the timing evidence (lag model, storm event study, negative controls in notebook 02) is solid, but a week-by-week national count forecast is only indicative. The app uses seasonal + rain because its effect of rain can be read and explained, and it shows this track record next to the forecast.

In [3]:
final = smf.negativebinomial(f'{BASE} + {RAIN}', d).fit(disp=0, maxiter=500)
coef = pd.DataFrame({'lag_weeks': list(FC_LAGS), 'coef': [final.params[f'share_l{L}'] for L in FC_LAGS]})
coef['unit'] = 'per 10 percentage points of provinces above their weekly 90th percentile'
coef.to_csv(OUT / 'forecast_coefs.csv', index=False)
# Overdispersion, used by the app for 80% ranges: variance = mean + alpha * mean^2
pd.DataFrame({'alpha': [final.params['alpha']]}).to_csv(OUT / 'forecast_alpha.csv', index=False)
coef.assign(alpha=final.params['alpha'])

,lag_weeks,coef,unit,alpha
0,2,0.014557,per 10 percentage points of provinces above th...,0.606228
1,3,0.130128,per 10 percentage points of provinces above th...,0.606228
2,4,0.020478,per 10 percentage points of provinces above th...,0.606228
3,5,0.071348,per 10 percentage points of provinces above th...,0.606228
4,6,0.074828,per 10 percentage points of provinces above th...,0.606228
